<a href="https://colab.research.google.com/github/flexfengfeng/SMU-Hackathon-Prep/blob/main/track2_social_innovation_silverpurpose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤝 Track 2 · Social Innovation
## SilverPurpose: Matching Seniors to Meaningful Ways to Stay Active and Contribute

**Coaching session · ~60 min**

### The sponsor's challenge
**raiSE**, Singapore's centre for social enterprise, asks:

> *"How might we support seniors in Singapore and India to age holistically, staying active and contributing to their communities, in ways that are affordable, accessible and motivating?"*

Out of scope: a solution that **depends on funding** (grants, donations) to keep running.

### The real-world problem
- Singapore became a **super-aged society** in 2026: 21.4% of citizens are 65 or older. India is ageing fast too.
- Ageing well is no longer just about living longer. It is about **connection, purpose and the chance to contribute**.
- Singapore has a national network of Active Ageing Centres, but participation is uneven. At some centres, **men are only 20% of participants**.
- Most programmes are **recreational** (karaoke, bingo, exercise) rather than purpose-driven. A retired electrician or teacher has decades of skill, but is offered a singing session.
- Most programmes need **paid staff and volunteers** to run, so they are costly to scale.

### The social-innovation angle
Stop treating seniors as **recipients of care**. Treat them as **people with skills that others need**.
SilverPurpose lets a senior describe themselves in their own words and language (typed or as a voice-note transcript). It then matches them to **paid gigs, volunteer roles and peer groups** that use what they already know: fixing, cooking, teaching, bookkeeping, storytelling.

It runs as a **social enterprise**: businesses, schools and companies **pay** for the seniors' time and skill, so it does not need grants. AI does the intake and matching, so one human coordinator can support many more seniors.

### What you will build
| Step | Technique | What it does |
|---|---|---|
| Understand | **Structured extraction (batched)** | Free text in any language ➜ a clean `SeniorProfile` (skills, mobility, days, language...) |
| Decide | **Transparent matching in plain Python** | Scores every opportunity with explainable reasons. *The AI does not decide.* |
| Explain | **Cross-lingual search (embeddings)** | Ask in Chinese, Hindi or Hinglish; find English opportunity descriptions |
| Respond | **Plain-language generation** | A short, large-text reply in the senior's own language, with one clear first step |
| Validate | **Pilot analytics with pandas** | Funnel, men's participation, earnings, loneliness change |
| Sustain | **Unit economics** | Revenue vs cost per match ➜ break-even, no grants needed |

> 🌏 **Languages in this notebook:** English (incl. Singlish), Chinese (中文) and Hindi (हिन्दी), including Hinglish.
>
> ⚠️ **Everything in this notebook is fictional:** the opportunities, organisations, people and pilot numbers. For your project, collect real data with consent and check facts against official sources.

## Step 1: Install Dependencies
- `google-genai`: Google's Gen AI SDK for Gemini (already preinstalled in Colab).
- `gradio`: the web UI for the demo.

`pandas`, `numpy`, `matplotlib` and `pydantic` are already in Colab.

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also set the client to wait and retry when the free tier says *"429: too many requests"*. Free-tier keys only allow about 5 requests per minute, so every team will hit this during the hackathon. If a cell seems to pause for up to a minute, it is waiting out the limit, not stuck.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

# Free-tier keys only allow a few requests per minute. These options make the SDK
# wait and retry any single request that gets "429: too many requests".
# We also save requests by batching: Step 4 reads three seniors' messages in ONE call.
RETRY = types.HttpRetryOptions(attempts=6, initial_delay=15, max_delay=60, http_status_codes=[429, 500, 503])

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY, http_options=types.HttpOptions(retry_options=RETRY))
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-3.5-flash-lite"

def ask_gemini(contents, config=None):
    """Call Gemini. Rate-limit retries are handled by the client (see RETRY above)."""
    return client.models.generate_content(model=MODEL, contents=contents, config=config)

## Step 3: The Opportunity Catalogue (Mock Data)
These are the ways a senior can stay active **and contribute**. Three kinds:
- **paid**: a small gig, e.g. bookkeeping for a shop or teaching a cooking class,
- **volunteer**: unpaid, but someone (a school, a company's CSR team) pays SilverPurpose to organise it,
- **peer group**: doing something together with other seniors.

Each opportunity has **machine-readable fields** (skills, physical demand, days, languages...) that Python matches on, plus a **human-readable description** that we search and explain.

Two fields matter for the business and the brief:
- `payer`: **who pays**. This is how the social enterprise earns money without grants.
- `purpose`: `contribution` or `recreation`. The brief says programmes today are mostly recreational. We include a few recreational ones on purpose, to show the matcher prefers roles with purpose.

Notice the roles built around **fixing, building, coaching and business advice**. Many men skip "activity centre" programmes but will say yes to *"we need your skills"*.

> 💡 **Hackathon tip:** your real catalogue could start as a Google Sheet of 20 opportunities you collected by phoning shops, schools and residents' groups. That phone-around is also customer validation.

In [ ]:
SKILLS = ["teaching", "mentoring", "storytelling", "cooking", "repair", "electrical", "carpentry", "gardening",
          "business", "bookkeeping", "tailoring", "tech", "sports", "music", "organising"]
LANGUAGES = ["English", "Mandarin", "Malay", "Tamil", "Hokkien", "Cantonese", "Hindi", "Marathi", "Punjabi"]
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri"]

def opp(id, title, country, area, type, skills, physical, days, time, languages, social, payer, pay, purpose, desc, remote=False):
    return dict(id=id, title=title, country=country, area=area, remote=remote, type=type, skills=skills, physical=physical,
                days=days, time=time, languages=languages, social=social, payer=payer, pay=pay, purpose=purpose, desc=desc)

# All organisations below are fictional. Amounts are in S$ (Singapore) or ₹ (India).
OPPORTUNITIES = [
    # ---- Singapore ----
    opp("SG01", "Repair Café fixer", "Singapore", "Bedok", "paid", ["repair", "electrical"], "medium", ["Sat"], "morning",
        ["English", "Malay", "Mandarin"], "social", "Town council repair programme", "S$40 per session", "contribution",
        "Fix fans, rice cookers, lamps and radios that residents bring to the monthly Bedok Repair Café, and show them how to do simple repairs."),
    opp("SG02", "Mentor for apprentice electricians", "Singapore", "Tampines", "paid", ["electrical", "mentoring", "teaching"], "low",
        ["Tue", "Thu"], "morning", ["English", "Malay"], "social", "EastBridge Training Institute (mentor fee)", "S$25 per hour", "contribution",
        "Coach young apprentice electricians on real-world wiring, safety habits and how to deal with customers. Sit-down workshop sessions."),
    opp("SG03", "Hawker recipe teacher", "Singapore", "Chinatown", "paid", ["cooking", "teaching"], "medium", ["Tue", "Wed", "Thu"], "afternoon",
        ["Mandarin", "Hokkien", "English"], "social", "Workshop participants and corporate team-building groups", "S$60 per class", "contribution",
        "Teach small groups to cook classic hawker dishes like laksa and lor mee, so family recipes are not lost. Chairs provided between steps."),
    opp("SG04", "Heritage storyteller and walk guide", "Singapore", "Chinatown", "paid", ["storytelling"], "high", ["Sat", "Sun"], "morning",
        ["English", "Mandarin", "Cantonese"], "social", "Little Lanes Tours (per tour)", "S$50 per tour", "contribution",
        "Lead 90-minute walking tours and share your own memories of how the neighbourhood used to be."),
    opp("SG05", "School reading buddy", "Singapore", "Ang Mo Kio", "volunteer", ["teaching", "storytelling"], "low", ["Mon", "Wed"], "morning",
        ["English", "Mandarin"], "social", "Primary school programme fee", "Unpaid (transport covered)", "contribution",
        "Read with one or two Primary 1 and 2 pupils who need extra help, once or twice a week, in the school library."),
    opp("SG06", "Community garden lead", "Singapore", "Toa Payoh", "volunteer", ["gardening", "organising"], "high", ["Mon", "Thu"], "morning",
        ["English", "Mandarin", "Malay"], "social", "Corporate CSR sponsor", "Unpaid", "contribution",
        "Run the estate's rooftop garden: plan planting, guide new volunteers and share harvests with neighbours."),
    opp("SG07", "Bookkeeping helper for small shops", "Singapore", "Jurong", "paid", ["bookkeeping", "business"], "low", WEEKDAYS, "afternoon",
        ["English", "Mandarin"], "solo", "Shop owners (hourly)", "S$18 per hour", "contribution",
        "Help family-run shops keep simple accounts, track invoices and prepare for tax filing. Can be done at the shop or from home."),
    opp("SG08", "Intergenerational tech-swap", "Singapore", "Bedok", "peer group", ["tech", "teaching", "cooking", "repair"], "low", ["Wed"], "afternoon",
        ["English", "Malay", "Mandarin"], "social", "Polytechnic service-learning programme fee", "Unpaid", "contribution",
        "Students teach seniors phone and app skills; seniors teach students a practical skill such as cooking or simple repairs."),
    opp("SG09", "Makers' shed: build furniture for families in need", "Singapore", "Tampines", "volunteer", ["carpentry", "repair"], "medium",
        ["Tue", "Fri"], "morning", ["English", "Mandarin", "Malay"], "social", "Corporate CSR partner (pays per piece built)", "Unpaid", "contribution",
        "Work with tools alongside other retirees to build and repair beds, shelves and study tables for low-income families."),
    opp("SG10", "Business advisor for young hawkers", "Singapore", "Islandwide", "paid", ["business", "cooking", "mentoring"], "low", WEEKDAYS, "flexible",
        ["Mandarin", "English", "Hokkien"], "social", "Hawker incubation programme fee", "S$30 per hour", "contribution",
        "Advise new hawkers on costing, suppliers, managing queues and running a stall. Meet at their stall or by video call.", remote=True),
    opp("SG11", "Clothes alteration micro-gigs", "Singapore", "Toa Payoh", "paid", ["tailoring"], "low", WEEKDAYS, "flexible",
        ["English", "Mandarin", "Malay"], "solo", "Customers through a partner laundry shop", "About S$8 per item", "contribution",
        "Do small alterations such as hemming and button repairs at home. The laundry shop collects and returns the clothes."),
    opp("SG12", "Walking group lead", "Singapore", "Bedok", "peer group", ["sports", "organising"], "high", ["Mon", "Wed", "Fri"], "morning",
        ["English", "Malay"], "social", "Small participant fee (S$2) and a health-insurer wellness partner", "S$10 per walk led", "contribution",
        "Lead a 45-minute neighbourhood walk for other seniors and check that everyone gets home safely."),
    opp("SG13", "Line dancing class", "Singapore", "Bedok", "peer group", ["music", "sports"], "medium", ["Tue", "Thu"], "afternoon",
        ["English", "Mandarin"], "social", "Participant fee (S$3)", "None", "recreation",
        "A friendly weekly line dancing class for seniors."),
    opp("SG14", "Karaoke and games afternoon", "Singapore", "Chinatown", "peer group", ["music"], "low", WEEKDAYS, "afternoon",
        ["Mandarin", "Cantonese", "Hokkien"], "social", "Free", "None", "recreation",
        "Sing old favourites and play bingo and board games with other seniors."),
    opp("SG15", "Online homework tutor", "Singapore", "Islandwide", "paid", ["teaching"], "low", WEEKDAYS, "evening",
        ["English", "Mandarin"], "solo", "Parents (SilverPurpose keeps 15%)", "S$20 per hour", "contribution",
        "Help primary-school children with homework over video call from home.", remote=True),
    # ---- India ----
    opp("IN01", "Online spoken-English and Hindi tutor", "India", "Pune", "paid", ["teaching"], "low",
        ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"], "morning", ["Hindi", "English", "Marathi"], "solo",
        "Parents and an ed-tech partner", "₹300 per hour", "contribution",
        "Teach spoken English or Hindi to school children over video call, from home, in small batches.", remote=True),
    opp("IN02", "Storytelling volunteer at a community preschool", "India", "Pune", "volunteer", ["storytelling", "teaching"], "low",
        ["Mon", "Wed", "Fri"], "morning", ["Hindi", "Marathi"], "social", "Corporate CSR programme fee", "Unpaid (auto fare covered)", "contribution",
        "Tell stories and read with children aged 3 to 6 at a preschool in your neighbourhood. You sit with the children in a small circle."),
    opp("IN03", "RWA seniors' committee member", "India", "Pune", "volunteer", ["organising", "business"], "low", ["Sat", "Sun"], "evening",
        ["Hindi", "Marathi", "English"], "social", "Resident Welfare Association (small honorarium)", "₹1,000 per month honorarium", "contribution",
        "Join your residents' association committee: plan events, run the safety phone tree and speak up for older residents."),
    opp("IN04", "Community kitchen (langar) coordinator", "India", "Delhi", "volunteer", ["cooking", "organising"], "medium", ["Sun"], "morning",
        ["Hindi", "Punjabi"], "social", "Gurdwara community kitchen trust", "Unpaid", "contribution",
        "Plan the Sunday community kitchen: ingredients, volunteer rota and food safety. Serves around 300 people."),
    opp("IN05", "Bookkeeping for kirana shops", "India", "Pune", "paid", ["bookkeeping", "business"], "low",
        ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"], "afternoon", ["Hindi", "Marathi"], "solo", "Shop owners (monthly)", "₹2,000 per shop per month", "contribution",
        "Keep simple accounts and GST records for neighbourhood grocery shops."),
    opp("IN06", "Skills mentor for electrician trainees", "India", "Mumbai", "paid", ["electrical", "repair", "mentoring"], "low",
        ["Tue", "Thu"], "morning", ["Hindi", "Marathi", "English"], "social", "Skilling NGO, funded by a company's CSR budget", "₹500 per session", "contribution",
        "Share real job experience with trainees at a vocational training institute: safety, fault-finding and dealing with customers."),
    opp("IN07", "Tailoring trainer for a women's self-help group", "India", "Delhi", "paid", ["tailoring", "teaching"], "low", ["Mon", "Wed"], "afternoon",
        ["Hindi"], "social", "Self-help group programme (CSR)", "₹400 per session", "contribution",
        "Teach stitching and finishing to women starting small tailoring businesses."),
    opp("IN08", "Laughter yoga club", "India", "Pune", "peer group", ["sports"], "medium", ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], "morning",
        ["Hindi", "Marathi"], "social", "Free", "None", "recreation",
        "Join a morning laughter yoga group in the park."),
    opp("IN09", "Old-city heritage walk guide", "India", "Pune", "paid", ["storytelling"], "high", ["Sat", "Sun"], "morning",
        ["Hindi", "Marathi", "English"], "social", "Heritage tour operator", "₹800 per walk", "contribution",
        "Guide visitors through the old city lanes and share stories of how life used to be."),
    opp("IN10", "Business mentor for women micro-entrepreneurs", "India", "Pune", "volunteer", ["business", "mentoring", "bookkeeping"], "low",
        ["Tue", "Thu"], "evening", ["Hindi", "English"], "solo", "Microfinance partner (programme fee)", "Unpaid", "contribution",
        "Phone or video mentoring for women running home businesses: pricing, record keeping and growing sales.", remote=True),
]

import pandas as pd
catalogue = pd.DataFrame(OPPORTUNITIES)
print(f"Loaded {len(catalogue)} opportunities.")
display(catalogue.groupby(["country", "type"]).size().unstack(fill_value=0))

## Step 4: Understand the Senior (Structured Extraction)
Seniors don't fill in long forms. They say things like *"Retired 2 years already lah, was electrician 35 years..."* or send a voice note. Gemini reads any language and fills in our `SeniorProfile` schema.

Coaching points:
- **Fixed vocabularies** (`Literal[...]`): skills, languages and days come back as tags from *our* list, so Python can match on them reliably.
- **"unknown" is allowed:** if the person didn't mention their mobility, the model must say `unknown` instead of **guessing**. Python then asks a coordinator to check.
- **Script, not just language:** someone who writes Hindi in English letters should get a reply in English letters too, so `reply_language` records the script.
- **Batching saves your rate limit:** we send all three seniors in **one** request and get a list of three profiles back.
- **Data minimisation:** notice what we do *not* ask for: no name, NRIC/Aadhaar, income, health conditions or gender. Only what we need to find a good match.

Meet our three fictional seniors:
| Senior | Where | Writes in |
|---|---|---|
| Mr Tan, retired electrician | Bedok, Singapore | English / Singlish |
| Madam Lee, former hawker | Chinatown, Singapore | Chinese (中文) |
| Mrs Joshi, retired schoolteacher | Pune, India | Hinglish (Hindi in English letters) |

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

Skill = Literal["teaching", "mentoring", "storytelling", "cooking", "repair", "electrical", "carpentry", "gardening",
                "business", "bookkeeping", "tailoring", "tech", "sports", "music", "organising"]
Language = Literal["English", "Mandarin", "Malay", "Tamil", "Hokkien", "Cantonese", "Hindi", "Marathi", "Punjabi"]
Day = Literal["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

class SeniorProfile(BaseModel):
    past_work: str = Field(description="Main past job or role in a few English words, or 'unknown'")
    skills: list[Skill] = Field(description="Skills from their work or life that they could use or teach")
    interests: list[Skill] = Field(description="Things they say they enjoy or would like to do")
    mobility: Literal["high", "medium", "low", "unknown"] = Field(
        description="high = can walk and stand for long; medium = some limits, e.g. can't stand long or climb; low = can't walk far")
    available_days: list[Day] = Field(description="Days they said they are free; empty list if not mentioned")
    preferred_time: Literal["morning", "afternoon", "evening", "any"]
    languages: list[Language] = Field(description="Languages they can speak comfortably")
    reply_language: str = Field(description="Language AND script they wrote in, e.g. 'English (Singlish)', "
                                            "'Chinese (Simplified)', 'Hindi in English letters (Hinglish)'")
    earn_or_give: Literal["earn", "give", "both", "unknown"] = Field(
        description="earn = wants paid work; give = happy to volunteer; both = either is fine")
    social_or_solo: Literal["social", "solo", "either"]
    wants_purpose: bool = Field(description="True if they want to be useful, pass on skills, or say they don't want purely recreational activities")
    country: Literal["Singapore", "India", "Unknown"]
    area: str = Field(description="Neighbourhood (Singapore) or city (India) where they live, or 'unknown'")

class ProfileBatch(BaseModel):
    profiles: list[SeniorProfile]

EXTRACT_PROMPT = """You help a social enterprise match seniors to paid gigs, volunteer roles and peer groups.
Read each senior's message (any language or mix of languages) and extract one profile per message, in the same order.
Only use facts the person stated or clearly implied. If something is not mentioned, use 'unknown', 'any', 'either',
or an empty list. Never guess. Skills include those from their past job (e.g. a former hawker can cook,
a teacher can teach). Return exactly one profile for each message.

"""

def extract_profiles(messages):
    numbered = "\n\n".join(f"Message {i + 1}:\n{m}" for i, m in enumerate(messages))
    response = ask_gemini(
        EXTRACT_PROMPT + numbered,
        types.GenerateContentConfig(response_mime_type="application/json", response_schema=ProfileBatch),
    )
    profiles = response.parsed.profiles
    if len(profiles) != len(messages):
        raise ValueError(f"Expected {len(messages)} profiles, got {len(profiles)}. Try again.")
    return profiles

PERSONAS = {
    "Mr Tan (Bedok)": "Retired 2 years already lah. Was electrician 35 years, HDB wiring, aircon, everything also can fix. "
                      "I stay Bedok. My wife ask me go the activity centre but I don't want to sit there sing karaoke and play bingo leh. "
                      "Legs still ok, just knee pain if climb ladder too long. Weekday mornings free. Got some pocket money also good, "
                      "but teach the young boys also can. English and Malay ok, Chinese a bit only.",
    "Madam Lee (Chinatown)": "我以前在牛车水卖了三十年的叻沙和卤面，去年收档了。孩子都大了，白天一个人在家很闷。我会讲华语、福建话，英文只会一点点。"
                             "脚还可以，就是不能站太久。想多认识一些朋友。星期二到星期四下午有空。赚不赚钱没关系，最想把手艺传下去，不要失传。",
    "Mrs Joshi (Pune)": "Namaste. Main Pune mein rehti hoon. 34 saal school mein Hindi aur English padhaya, ab retired hoon. "
                        "Ghar pe akele time nahi katta. Bachchon ko padhana aur kahaniyan sunana bahut pasand hai. "
                        "Smartphone chalana aata hai, online bhi padha sakti hoon. Thoda extra income mile toh accha, warna seva bhi theek hai. "
                        "Ghutno ki wajah se zyada chal nahi paati. Monday se Saturday subah free hoon.",
}

# ONE request for all three seniors
profiles = dict(zip(PERSONAS, extract_profiles(list(PERSONAS.values()))))
for name, p in profiles.items():
    print(f"👤 {name}\n{p.model_dump_json(indent=2)}\n")

## Step 5: Match in Plain Python (LLM Perceives, Python Decides)
**Coaching point:** we do *not* ask the LLM *"what should Mr Tan do?"*. The match decides where a senior spends their time and who they meet, so it must be:
- **consistent**: the same profile always gets the same matches,
- **explainable**: we can say exactly *why* each match was suggested,
- **tunable**: the team (and later, seniors' feedback) can change the weights in one place.

The matcher works in two stages:
1. **Hard filters** (cannot do): wrong country, an Indian city too far away (unless online), too physically demanding, or no shared language.
2. **Score out of 100**, made of explainable parts:

| Part | Max | Idea |
|---|---|---|
| Skills | 30 | Uses what they already know (past job counts most) |
| Purpose | 15 | Contribution beats pure recreation; recreation is penalised if they said they don't want it |
| Earn / give | 15 | Paid gig if they want income, volunteering if they want to give |
| Schedule | 10 | Matches their free days and time of day |
| Language | 10 | They can speak the group's language |
| Social / solo | 10 | Group or alone, as they prefer |
| Location | 10 | Same neighbourhood or city; online roles get most points |

In [ ]:
LEVEL = {"low": 0, "medium": 1, "high": 2}
EARN_GIVE_POINTS = {  # preference -> {opportunity type: points}
    "earn":    {"paid": 15, "peer group": 5,  "volunteer": 0},
    "give":    {"paid": 5,  "peer group": 10, "volunteer": 15},
    "both":    {"paid": 15, "peer group": 8,  "volunteer": 15},
    "unknown": {"paid": 8,  "peer group": 8,  "volunteer": 8},
}

def hard_filter(o, p):
    """Return a reason if this opportunity is not possible for this person, else None."""
    if p.country != "Unknown" and o["country"] != p.country:
        return f"in {o['country']}"
    if o["country"] == "India" and not o["remote"] and o["area"].lower() != p.area.lower():
        return f"in {o['area']}, not online"
    mobility = "medium" if p.mobility == "unknown" else p.mobility
    if LEVEL[o["physical"]] > LEVEL[mobility]:
        return f"physically {o['physical']} demand"
    if not set(o["languages"]) & set(p.languages):
        return "no shared language"
    return None

def score_opportunity(o, p):
    score, why = 0, []
    skill_hits = set(o["skills"]) & set(p.skills)
    interest_hits = (set(o["skills"]) & set(p.interests)) - skill_hits
    s = min(30, 20 * len(skill_hits) + 10 * len(interest_hits))
    score += s
    if skill_hits or interest_hits:
        why.append(f"uses your {', '.join(sorted(skill_hits | interest_hits))}")

    if o["purpose"] == "contribution":
        score += 15
        why.append("you contribute, not just attend")
    elif p.wants_purpose:
        score -= 10
        why.append("⚠️ purely recreational, which you said you don't want")

    score += EARN_GIVE_POINTS[p.earn_or_give][o["type"]]
    if o["type"] == "paid" and p.earn_or_give in ("earn", "both"):
        why.append(f"paid: {o['pay']}")
    elif o["type"] == "volunteer" and p.earn_or_give in ("give", "both"):
        why.append("volunteer role, as you'd like")

    day_ok = not p.available_days or bool(set(o["days"]) & set(p.available_days))
    time_ok = p.preferred_time == "any" or o["time"] in ("flexible", p.preferred_time)
    score += 5 * day_ok + 5 * time_ok
    if not p.available_days:
        why.append("ℹ️ free days not given, coordinator to check")
    elif day_ok and time_ok:
        why.append(f"fits your free time ({'/'.join(o['days'][:3])} {o['time']})")
    else:
        why.append(f"⚠️ schedule: {'/'.join(o['days'][:3])} {o['time']}")

    score += 10  # shared language is guaranteed by the hard filter
    why.append(f"in {'/'.join(sorted(set(o['languages']) & set(p.languages)))}")

    if p.social_or_solo in ("either", o["social"]):
        score += 10
    if o["area"].lower() == p.area.lower():
        score += 10
        why.append(f"near you in {o['area']}")
    elif o["remote"]:
        score += 8
        why.append("can do from home / online")
    else:
        score += 6  # same country (Singapore is small)
    if p.mobility == "unknown":
        why.append("ℹ️ mobility not given, coordinator to check")
    return score, why

def match(p, top_n=3):
    ranked, excluded = [], []
    for o in OPPORTUNITIES:
        reason = hard_filter(o, p)
        if reason:
            excluded.append((o["id"], reason))
            continue
        s, why = score_opportunity(o, p)
        ranked.append({"id": o["id"], "title": o["title"], "type": o["type"], "area": o["area"], "score": s, "why": "; ".join(why)})
    ranked.sort(key=lambda r: -r["score"])
    return ranked[:top_n], excluded

matches = {}
for name, p in profiles.items():
    matches[name], excluded = match(p)
    print(f"👤 {name}: {len(excluded)} opportunities filtered out, e.g. {excluded[:2]}")
    display(pd.DataFrame(matches[name]))

## Step 6: Cross-Lingual Search for Follow-up Questions
After seeing their matches, seniors (or their children) ask follow-up questions: *"有没有教人煮菜的机会？"* ("Is there anything teaching people to cook?"), *"क्या घर से कोई काम हो सकता है?"* ("Can I do any work from home?").

Gemini's embedding model is **multilingual**: a question in Chinese or Hindi lands close to the matching English description in vector space, so we don't need to translate the catalogue into every language.

**Why no vector database here?** The old way was to load the documents into ChromaDB. With 25 opportunities, a NumPy array and one line of maths (cosine similarity) does the same job, with nothing extra to install or break in Colab. Move to a vector database (ChromaDB, pgvector...) when your catalogue grows to thousands of items or you need to save it between sessions.

We also embed all 25 descriptions in **one** request, and all test questions in **one** more request.

In [ ]:
import numpy as np

EMBED_MODEL = "gemini-embedding-001"

def embed(texts, task_type):
    result = client.models.embed_content(model=EMBED_MODEL, contents=texts,
                                         config=types.EmbedContentConfig(task_type=task_type))
    vectors = np.array([e.values for e in result.embeddings])
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)  # unit length, so dot product = cosine similarity

def opp_text(o):
    where = f"{o['area']}, {o['country']}" + (" (online / from home)" if o["remote"] else "")
    return (f"{o['title']} ({where}). {o['desc']} Type: {o['type']}. Pay: {o['pay']}. "
            f"When: {', '.join(o['days'])} {o['time']}. Languages: {', '.join(o['languages'])}.")

OPP_TEXTS = [opp_text(o) for o in OPPORTUNITIES]
OPP_VECTORS = embed(OPP_TEXTS, "RETRIEVAL_DOCUMENT")  # one request for the whole catalogue

def search(question=None, n=3, country=None, query_vector=None):
    q = embed([question], "RETRIEVAL_QUERY")[0] if query_vector is None else query_vector
    sims = OPP_VECTORS @ q
    order = [i for i in np.argsort(-sims) if country in (None, OPPORTUNITIES[i]["country"])]
    return [(OPPORTUNITIES[i]["id"], round(float(sims[i]), 3), OPP_TEXTS[i]) for i in order[:n]]

questions = [
    ("有没有教人煮菜的机会？", "Singapore"),                         # Chinese: any chance to teach cooking?
    ("क्या घर से कोई काम हो सकता है?", "India"),                    # Hindi: can I do any work from home?
    ("Koi seva wala kaam hai jisme bachchon ke saath rehna ho?", "India"),  # Hinglish: volunteering with children?
    ("Got anything where I can fix things one?", "Singapore"),       # Singlish
]
query_vectors = embed([q for q, _ in questions], "RETRIEVAL_QUERY")  # one request for all questions
for (q, country), v in zip(questions, query_vectors):
    top_id, sim, text = search(country=country, query_vector=v, n=1)[0]
    print(f"❓ {q}\n   ➜ {top_id} ({sim}): {text[:90]}...\n")

In [ ]:
def answer_question(question, country=None):
    hits = search(question, n=3, country=country)
    info = "\n".join(f"- [{i}] {t}" for i, _, t in hits)
    prompt = ("""You are SilverPurpose, a respectful assistant for seniors. Answer in the same language and script as the question,
in simple words and short sentences. Use ONLY the information below and mention the opportunity names.
If the answer isn't there, say a SilverPurpose coordinator can help.\n\nInformation:\n""" + info + f"\n\nQuestion: {question}")
    return ask_gemini(prompt).text, hits

answer, _ = answer_question(*questions[0])
print(answer)

## Step 7: Reply in Simple Words, in the Senior's Language
The reply combines:
- the **matches from Python**, which are the facts (the model may not change or add any),
- **style rules**: their language and script, short lines, no jargon, large text,
- **dignity**: talk to them as someone whose skills are *needed*, not as someone who needs help, and
- **one clear first step**: *"Reply YES and a coordinator will call you"*, or a WhatsApp group link.

Once more we batch: **one** request writes all three replies. In the app, it is one request per senior.

In [ ]:
import html
from IPython.display import HTML

WHATSAPP_LINK = "https://chat.whatsapp.com/YOUR-GROUP-LINK"  # placeholder: create a real group for your pilot

class Replies(BaseModel):
    replies: list[str] = Field(description="One reply per senior, in the same order")

REPLY_PROMPT = """You are SilverPurpose, a social enterprise that connects seniors with roles where their skills are needed.
Write one WhatsApp reply for each senior below, in the same order.

Rules for every reply:
- Write in the senior's language AND script (given for each senior). Very simple words, short lines, max 90 words.
- Speak to them as a respected person with valuable experience. Never as someone who needs care or charity.
- Start with one warm line about their experience. Then list the matches as 1, 2, 3: name, when/where, pay if any.
- Only use the matches given. Do NOT change, add or rank them differently.
- End with this first step, translated: "Reply YES and a SilverPurpose coordinator will call you within 2 days.
  Or join our WhatsApp group: {link}"
"""

def describe_matches(top):
    lines = []
    for r in top:
        o = next(o for o in OPPORTUNITIES if o["id"] == r["id"])
        lines.append(f"  - {o['title']} | {o['area']} | {'/'.join(o['days'])} {o['time']} | pay: {o['pay']} | "
                     f"what: {o['desc']} | why it fits: {r['why']}")
    return "\n".join(lines)

def write_replies(items):
    """items: list of (message, profile, top_matches). One request for all of them."""
    blocks = [f"Senior {i + 1}\n- Write in: {p.reply_language}\n- Their message: {m}\n- Matches:\n{describe_matches(top)}"
              for i, (m, p, top) in enumerate(items)]
    response = ask_gemini(
        REPLY_PROMPT.format(link=WHATSAPP_LINK) + "\n\n" + "\n\n".join(blocks),
        types.GenerateContentConfig(response_mime_type="application/json", response_schema=Replies),
    )
    return response.parsed.replies

def large_text(reply):
    """Big, high-contrast text: many seniors struggle with small fonts."""
    body = html.escape(reply).replace("\n", "<br>")
    return (f'<div style="font-size:22px;line-height:1.6;background:#fffbe6;color:#222;padding:16px;'
            f'border-radius:12px;max-width:640px">{body}</div>')

replies = write_replies([(PERSONAS[n], profiles[n], matches[n]) for n in PERSONAS])
for name, reply in zip(PERSONAS, replies):
    print(f"📱 To {name}:")
    display(HTML(large_text(reply)))

### 🤔 Coaching discussion (5 min)
- Look at Mr Tan's matches. Did the **karaoke afternoon** and **line dancing** drop to the bottom? Which part of the score did that? What would *you* offer a retired man who "doesn't want to sit there sing karaoke"?
- Mrs Joshi has bad knees. Which opportunities did the hard filter remove, and is that **respectful** or **patronising**? Should she be allowed to override it?
- Change Madam Lee's message so she doesn't mention her free days. Do you see the "coordinator to check" note instead of a guess?
- The weights (30 for skills, 15 for purpose...) are **our opinion**. How would you learn better weights from the pilot data in Step 8?
- Read the replies aloud. Do they sound like *"we need you"* or like *"poor thing, come join our activity"*? Which words make the difference?
- Who is harmed if a match is wrong: the senior who travels for nothing, or a shop that receives an unsuitable helper? How does the coordinator call prevent that?

## Step 8: Validate Interest and Impact with a Pilot Log
raiSE's strong outcome includes **ways to validate market and customer interest**. An AI demo doesn't prove seniors will turn up. A small pilot does.

Below is a **synthetic** log of an imagined 8-week pilot with 200 seniors in Singapore and India. Pandas turns it into the numbers judges ask about:
- **Funnel:** signed up ➜ matched ➜ attended first session ➜ still active after 4 weeks.
- **Men's participation**, compared with the 20% seen at some Active Ageing Centres.
- **Share of paid gigs** and **earnings** (seniors' extra income).
- **Loneliness change** on the short 3-item UCLA Loneliness Scale (3 = least lonely, 9 = most).
- **Willingness to pay** for a premium workshop, a first signal that seniors value it.

> ⚠️ **Illustrative only.** These numbers are invented to show the analysis. Your real pilot (even 20 seniors over 2 weeks) replaces them. Gender here is an optional, self-reported sign-up field used only to measure reach, never for matching.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
N = 200
pilot = pd.DataFrame({
    "senior_id": [f"P{i:03d}" for i in range(N)],
    "country": rng.choice(["Singapore", "India"], N),
    "gender": rng.choice(["male", "female"], N, p=[0.45, 0.55]),
    "channel": rng.choice(["WhatsApp", "Community poster", "Residents' group", "Friend referral"], N, p=[0.35, 0.2, 0.2, 0.25]),
})
pilot["signed_up"] = True
pilot["matched"] = rng.random(N) < 0.82
pilot["match_type"] = np.where(pilot.matched, rng.choice(["paid", "volunteer", "peer group"], N, p=[0.4, 0.45, 0.15]), "none")
pilot["attended_first"] = pilot.matched & (rng.random(N) < np.where(pilot.match_type == "paid", 0.8, 0.7))
pilot["active_week4"] = pilot.attended_first & (rng.random(N) < 0.72)
earn = np.where(pilot.country == "Singapore", rng.normal(220, 60, N), rng.normal(45, 15, N)).clip(10).round()  # S$ equivalent
pilot["earnings_sgd_4wk"] = np.where(pilot.active_week4 & (pilot.match_type == "paid"), earn, 0)
pilot["loneliness_before"] = rng.integers(4, 10, N)
improve = np.where(pilot.active_week4, rng.choice([0, 1, 1, 2], N), rng.choice([0, 0, 1], N))
pilot["loneliness_after"] = (pilot.loneliness_before - improve).clip(3, 9)
pilot["would_pay_premium"] = rng.random(N) < np.where(pilot.active_week4, 0.45, 0.15)
display(pilot.head())

In [ ]:
STAGES = ["signed_up", "matched", "attended_first", "active_week4"]
funnel = pilot.groupby("gender")[STAGES].sum().T
funnel["total"] = funnel.sum(axis=1)
funnel["% of sign-ups"] = (100 * funnel["total"] / N).round(1)
funnel["men %"] = (100 * funnel["male"] / funnel["total"]).round(1)
display(funnel)

active = pilot[pilot.active_week4]
matched = pilot[pilot.matched]
paid_active = active[active.match_type == "paid"]
lonely = pilot.assign(change=pilot.loneliness_after - pilot.loneliness_before).groupby("active_week4")["change"].mean()

KPIS = {
    "Active after 4 weeks": f"{len(active)} of {N} ({len(active) / N:.0%})",
    "Men among active seniors": f"{(active.gender == 'male').mean():.0%} (baseline at some centres: 20%)",
    "Matches that are paid gigs": f"{(matched.match_type == 'paid').mean():.0%}",
    "Median 4-week earnings, paid gigs": ", ".join(f"{c}: S${v:,.0f}" for c, v in paid_active.groupby("country")["earnings_sgd_4wk"].median().items()),
    "Loneliness change (active)": f"{lonely.get(True, 0):+.2f} points",
    "Loneliness change (not active)": f"{lonely.get(False, 0):+.2f} points",
    "Active seniors who would pay for a premium workshop": f"{active.would_pay_premium.mean():.0%}",
}
for k, v in KPIS.items():
    print(f"{k:<55} {v}")

In [ ]:
def pilot_figure():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    funnel[["female", "male"]].plot.bar(ax=axes[0], color=["#e07a5f", "#3d405b"], rot=0)
    axes[0].set_title("Pilot funnel by gender (illustrative)")
    axes[0].set_ylabel("Seniors")
    axes[0].set_xticks(range(len(STAGES)), ["Signed up", "Matched", "Attended 1st", "Active wk 4"])
    means = pilot.groupby("active_week4")[["loneliness_before", "loneliness_after"]].mean()
    means.index = ["Not active", "Active wk 4"]
    means.plot.bar(ax=axes[1], color=["#bbbbbb", "#81b29a"], rot=0)
    axes[1].set_title("Mean loneliness score, 3 (low) to 9 (high), illustrative")
    axes[1].legend(["Before", "After 4 weeks"])
    fig.tight_layout()
    return fig

pilot_figure()
plt.show()

## Step 9: The Social Enterprise Model (No Grants Needed)
raiSE rules out solutions that **depend on funding**. So the question is: *who pays, and how many matches a month do we need to cover our costs?*

Revenue streams, all **payments for a service**, not donations:
- **Placement fee** from businesses that hire a senior for a paid gig (shops, training institutes, tour companies).
- **Programme fee** from schools and companies' CSR teams for organising volunteer roles (reading buddies, makers' shed). This is a service contract: they get a well-run programme and impact reporting.
- **Premium workshop fee**, a small fee some seniors pay, e.g. for a cooking masterclass. Peer groups stay free.

Costs per match: the coordinator's phone call and check-in (AI does the intake and shortlist, so this is about 1 hour instead of 3), a safety check of the business or school, insurance, and messaging/AI costs.

The match mix (paid vs volunteer vs peer group) comes from the pilot in Step 8. All prices below are **assumptions to test**.

In [ ]:
mix = matched.match_type.value_counts(normalize=True)  # share of paid / volunteer / peer-group matches from the pilot

ASSUMPTIONS = {
    "Singapore": dict(cur="S$", placement_fee=150, programme_fee=120, premium_fee=15, premium_uptake=0.3,
                      coordinator_cost=40, safety_check=10, insurance_and_tools=5, fixed_monthly=4000),
    "India":     dict(cur="₹",  placement_fee=1500, programme_fee=1200, premium_fee=100, premium_uptake=0.3,
                      coordinator_cost=300, safety_check=100, insurance_and_tools=50, fixed_monthly=60000),
}

def unit_economics(a):
    revenue = {
        "Placement fee (paid gigs)": a["placement_fee"] * mix.get("paid", 0),
        "Programme fee (school / CSR)": a["programme_fee"] * mix.get("volunteer", 0),
        "Premium workshop fee (seniors)": a["premium_fee"] * a["premium_uptake"],
    }
    cost = {k: a[k] for k in ["coordinator_cost", "safety_check", "insurance_and_tools"]}
    margin = sum(revenue.values()) - sum(cost.values())
    break_even = int(np.ceil(a["fixed_monthly"] / margin)) if margin > 0 else None
    return revenue, cost, margin, break_even

rows = []
for country, a in ASSUMPTIONS.items():
    revenue, cost, margin, break_even = unit_economics(a)
    rows += [(country, "Revenue", k, f"{a['cur']}{v:,.0f}") for k, v in revenue.items()]
    rows += [(country, "Cost", k.replace("_", " "), f"-{a['cur']}{v:,.0f}") for k, v in cost.items()]
    rows.append((country, "=", "Contribution per match", f"{a['cur']}{margin:,.0f}"))
    rows.append((country, "=", f"Break-even (fixed costs {a['cur']}{a['fixed_monthly']:,}/month)",
                 f"{break_even} matches/month" if break_even else "never: fix the prices"))
display(pd.DataFrame(rows, columns=["Country", "", "Item (average per match)", "Amount"]))
print(f"Match mix from the pilot: {mix.round(2).to_dict()}")

## Step 10: Build the Demo UI with Gradio 🏆
Three tabs:
1. **Find my purpose**: a senior (or their child) types or pastes a voice-note transcript in any language and picks a city. Out come the matches with reasons, and the reply in large text. *Two* Gemini requests per click (read + reply).
2. **Pilot dashboard**: the funnel, KPIs and a break-even calculator with sliders. No AI at all, so no rate limit.
3. **Ask a question**: the cross-lingual search from Step 6. One embedding request and one Gemini request.

In [ ]:
import gradio as gr

AREA_COUNTRY = {"Bedok": "Singapore", "Tampines": "Singapore", "Chinatown": "Singapore", "Ang Mo Kio": "Singapore",
                "Toa Payoh": "Singapore", "Jurong": "Singapore", "Pune": "India", "Mumbai": "India", "Delhi": "India"}
FROM_MESSAGE = "(from my message)"

def ui_match(message, city):
    if not message.strip():
        return {}, [], large_text("Please tell us a little about yourself: your past work, what you enjoy, and when you are free.")
    try:
        p = extract_profiles([message])[0]
        if city != FROM_MESSAGE:
            p = p.model_copy(update={"area": city, "country": AREA_COUNTRY[city]})
        top, _ = match(p)
        if not top:
            return p.model_dump(), [], large_text("Thank you! A SilverPurpose coordinator will call you to find a good fit.")
        reply = write_replies([(message, p, top)])[0]
    except Exception as e:
        return {}, [], f"⚠️ Sorry, something went wrong: {e}"
    table = [[r["title"], r["type"], r["score"], r["why"]] for r in top]
    return p.model_dump(), table, large_text(reply)

def ui_break_even(country, placement_fee, programme_fee, coordinator_cost):
    a = {**ASSUMPTIONS[country], "placement_fee": placement_fee, "programme_fee": programme_fee, "coordinator_cost": coordinator_cost}
    _, _, margin, break_even = unit_economics(a)
    if not break_even:
        return f"### ❌ Every match loses {a['cur']}{-margin:,.0f}. Raise fees or cut costs."
    return (f"### Contribution per match: {a['cur']}{margin:,.0f}\n"
            f"### Break-even: **{break_even} matches per month** (fixed costs {a['cur']}{a['fixed_monthly']:,})")

def ui_ask(question, country):
    if not question.strip():
        return ""
    try:
        answer, _ = answer_question(question, None if country == "Any" else country)
        return answer
    except Exception as e:
        return f"⚠️ Sorry, something went wrong: {e}"

with gr.Blocks(title="SilverPurpose") as demo:
    gr.Markdown("# 🌱 SilverPurpose\nYour experience is needed. Tell us about yourself in any language.\n"
                "*Demo with fictional opportunities. A human coordinator confirms every match.*")
    with gr.Tab("Find my purpose"):
        with gr.Row():
            msg_in = gr.Textbox(label="About you (type, or paste a voice-note transcript)", lines=5, scale=3)
            city_in = gr.Dropdown([FROM_MESSAGE] + list(AREA_COUNTRY), value=FROM_MESSAGE, label="Where you live", scale=1)
        gr.Examples([[m, FROM_MESSAGE] for m in PERSONAS.values()], inputs=[msg_in, city_in])
        go = gr.Button("Find my matches", variant="primary")
        reply_out = gr.HTML()
        with gr.Accordion("For coordinators: how we matched", open=False):
            table_out = gr.Dataframe(headers=["Opportunity", "Type", "Score", "Why"], wrap=True)
            profile_out = gr.JSON(label="Extracted profile")
        go.click(ui_match, inputs=[msg_in, city_in], outputs=[profile_out, table_out, reply_out])
    with gr.Tab("Pilot dashboard"):
        gr.Markdown("**Illustrative pilot data.** Replace it with your own pilot log.\n\n" + "\n".join(f"- **{k}:** {v}" for k, v in KPIS.items()))
        gr.Dataframe(funnel.reset_index(names="stage"), label="Funnel")
        gr.Plot(pilot_figure())
        gr.Markdown("### Break-even calculator")
        with gr.Row():
            be_country = gr.Radio(list(ASSUMPTIONS), value="Singapore", label="Country")
            be_place = gr.Slider(0, 3000, value=150, step=10, label="Placement fee per paid gig")
            be_prog = gr.Slider(0, 3000, value=120, step=10, label="Programme fee per volunteer match")
            be_coord = gr.Slider(0, 1000, value=40, step=5, label="Coordinator cost per match")
        be_out = gr.Markdown(ui_break_even("Singapore", 150, 120, 40))
        # Switching country loads that country's default prices into the sliders
        be_country.change(lambda c: [ASSUMPTIONS[c][k] for k in ("placement_fee", "programme_fee", "coordinator_cost")],
                          inputs=be_country, outputs=[be_place, be_prog, be_coord])
        for ctrl in [be_place, be_prog, be_coord]:
            ctrl.change(ui_break_even, inputs=[be_country, be_place, be_prog, be_coord], outputs=be_out)
    with gr.Tab("Ask a question"):
        q_in = gr.Textbox(label="Your question (any language)", placeholder="e.g. 有没有教人煮菜的机会？ / क्या घर से कोई काम हो सकता है?")
        q_country = gr.Radio(["Any", "Singapore", "India"], value="Any", label="Country")
        ask_btn = gr.Button("Ask")
        a_out = gr.Markdown()
        ask_btn.click(ui_ask, inputs=[q_in, q_country], outputs=a_out)

# Launch it! (This will generate a public URL you can share with judges)
demo.launch(share=True, theme="soft")

## 🎯 What the Sponsor Wants After 36 Hours
raiSE's strong outcome: **a clear understanding of the issue or pain point, the potential impact, and ways to validate market/customer interest.** A demo alone is not enough. Bring:

- [ ] A sharp **problem / pain-point statement**, ideally backed by 5+ conversations with real seniors or caregivers.
- [ ] An **existing-solutions scan**, showing how your idea is *different* (raiSE puts this in scope explicitly).
- [ ] Your idea mapped to the **raiSE Social Impact Framework**: Intentionality, Additionality, Proportionality.
- [ ] A **Social Enterprise Model Canvas**: desirability, feasibility, viability.
- [ ] A **validation plan** for seniors (users) *and* for the businesses and schools who pay (customers).
- [ ] A business model that does **not** depend on grants or donations (Step 9).
- [ ] Coverage of **both Singapore and India**.

Copy these templates into your pitch notes and fill them in.

**1. Problem / pain-point statement**
> *[Who]* (e.g. retired men aged 60–75 in Bedok who left skilled trades) *struggle to [pain]* (find a reason to leave the house that uses their skills) *because [root cause]* (most programmes are recreational and run by staff, and nobody asks what they can offer). *This leads to [consequence]* (isolation, lost income, faster decline). *We know because [evidence]* (interviews, data).

**2. Existing-solutions scan** (the first rows are starting points only: verify every detail from official sources)
| Solution | Country | What it offers | Who pays | Gap SilverPurpose fills |
|---|---|---|---|---|
| Active Ageing Centres | Singapore | Activities, befriending, care referrals near home | Government-funded operators | *(verify; e.g. mostly recreational, low male participation)* |
| RSVP Singapore (The Organisation of Senior Volunteers) | Singapore | Senior volunteering programmes | *(verify)* | *(verify)* |
| *(e.g. a senior jobs portal or re-employment scheme)* | Singapore / India | | | |
| *(e.g. senior citizens' clubs, RWAs, temple/gurdwara seva)* | India | | | |
| *(add 2–3 more you find)* | | | | |

**3. raiSE Social Impact Framework**
| Lens | Question to answer | Your answer |
|---|---|---|
| **Intentionality** | Is social impact the *purpose* of the business, built into the model, not a side effect? Who exactly is the beneficiary? | |
| **Additionality** | What happens *because of you* that would not happen anyway? (e.g. matches for men who would never attend a centre) | |
| **Proportionality** | Is the impact meaningful relative to the money and effort, and do you measure it honestly? Which 2–3 metrics? | |

**4. Social Enterprise Model Canvas** (summary)
| Lens | Key questions | Your answer |
|---|---|---|
| **Desirability** | Do seniors want this? Do businesses/schools want senior talent? What evidence do you have? | |
| **Feasibility** | Can you build and run it? Partners, coordinators, safety checks, languages, channels (WhatsApp, voice)? | |
| **Viability** | Who pays, how much, and at how many matches per month do you break even? | |

**5. Validation plan**
| Hypothesis | Cheapest test this month | Metric | Pass if |
|---|---|---|---|
| Seniors sign up when offered purpose, not activities | WhatsApp sign-up form shared via 2 residents' groups | Sign-ups in 2 weeks | ≥ 30 |
| Men join at a higher rate than centres | Same form, optional gender question | % men among sign-ups | ≥ 35% |
| Matches stick | Coordinator check-in at week 4 | % still active | ≥ 50% |
| Businesses will pay a placement fee | Phone 20 shops / schools with a one-page offer | Letters of intent | ≥ 3 |
| Seniors value it | Offer a premium workshop | % who pay | ≥ 20% |

## ✅ Constraints Checklist
| Brief requirement | How this prototype handles it | Status |
|---|---|---|
| Seniors in **Singapore and India** | Catalogue, personas and economics for both | ✅ |
| **Holistic**: active, connected, contributing | Purpose-weighted matching, social preference, loneliness metric | ✅ |
| **Affordable** | Free for seniors; businesses, schools and CSR teams pay | ✅ (prices are assumptions) |
| **Accessible** | Any language and script, voice-note transcripts, WhatsApp-style reply, large text | ✅ (voice input itself is an extension) |
| **Motivating** | Roles that use their skills, some paid; "your experience is needed" tone | ✅ (validate in pilot) |
| Not **manpower-intensive** | AI does intake, matching and replies; coordinator only confirms (~1 h per match) | ⚠️ needs real time-tracking |
| **No dependency on funding** | Break-even model from service fees, no grants | ✅ on paper, ⚠️ CSR fees are a concentration risk |
| Reaches **men** (20% at some centres) | Fixing, building, coaching, business roles; men's share tracked | ⚠️ synthetic pilot only |
| **Different from existing solutions** | Skills-first matching + paid gigs + no-staff intake | ⚠️ complete the scan above |
| Clear **pain point, impact, validation** | Templates + pilot analytics | ⚠️ your team fills in with real evidence |

## 🚀 Make It Your Own: Hackathon Extensions
Pick one or two. Don't try all of them.
- **Voice in, voice out:** Gemini understands audio, so a senior can send a WhatsApp voice note in Hokkien or Marathi. Add text-to-speech for the reply.
- **Real channel:** a Telegram or WhatsApp Business bot, so seniors never open a web page.
- **Demand side:** a form where shops, schools and companies post opportunities; the same extraction turns their messy text into catalogue rows.
- **Learn the weights:** use pilot outcomes (still active at week 4?) to tune the matching weights, keeping them readable.
- **Buddy matching:** pair two seniors for the same role, so the first visit is less scary.
- **Skills passport:** a simple record of roles done and thank-you notes, which seniors can show to the next employer or family.

## 🎤 Pitching to Judges
- Open with **one person**: *"Mr Tan fixed HDB wiring for 35 years. Today he's offered karaoke."* Then show him matched to mentoring apprentices, paid, in 30 seconds.
- Demo **three languages** live: Singlish, Chinese and Hinglish.
- Show the **coordinator panel**: *"AI understands, Python decides, a human confirms."*
- Show the **break-even number** and **who pays**. raiSE wants a business, not a grant application.
- Be honest: *"These pilot numbers are illustrative. Here is our plan to get real ones in 4 weeks."*

## 🛡️ Responsible AI Checklist
- [ ] **Consent:** seniors agree before their details are shared with a business or school; they can withdraw anytime. If a child fills it in for a parent, confirm with the parent.
- [ ] **Dignity:** language treats seniors as contributors, not recipients of care. No "poor thing" tone; test replies with real seniors.
- [ ] **Safety for paid gigs:** verify every business and school, pay rates are shown up front, no upfront fees from seniors ever, and a coordinator checks in after the first session. Warn about scams that pretend to offer jobs.
- [ ] **Ageism and fairness:** the matcher must not push women into cooking and men into fixing. Check matches by gender and age group; let seniors override filters (e.g. mobility).
- [ ] **Data minimisation:** collect only what matching needs (no NRIC/Aadhaar, income or health details). Follow Singapore's **PDPA** and India's **Digital Personal Data Protection (DPDP) Act**; delete chats you don't need.
- [ ] **Human coordinator in the loop:** every match is confirmed by a person before anyone is introduced.
- [ ] The free Gemini API tier may use your inputs to improve Google's products, so **never test with real people's data**.